# **[DEPERECATED]**

# Реранкинг и лучшая комбинация каналов

In [1]:
import sys

sys.path.append("..")

from collections import defaultdict

import bm25s
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from eval import DEFAULT_SEARCH_COLS, make_val_split, recall_at_50
from preprocessing import clean_text, tokenize

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)

print("train_part:", len(train_part), "строк")
print("val_queries:", len(val_queries), "запросов")

train_part: 448040 строк
val_queries: 35447 запросов


## Dense: item=enriched, query=plain

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

corpus_embeddings = np.load("../data/embeddings/e5_base_enriched/512.npy")
dense_item_ids = np.load("../data/embeddings/e5_base_enriched/512_item_ids.npy", allow_pickle=True).tolist()
print("item-эмбеды (enriched):", corpus_embeddings.shape)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

embed_model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
embed_model.max_seq_length = 512

val_query_texts = val_queries["search_query"].apply(clean_text).apply(lambda t: f"query: {t}").tolist()
query_embeddings = embed_model.encode(
    val_query_texts, batch_size=64, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
)

_, indices = index.search(query_embeddings, 50)

dense_predictions = {
    cid: [dense_item_ids[i] for i in row if i != -1]
    for row, cid in zip(indices, val_queries["context_id"])
}
dense_recall = recall_at_50(dense_predictions, val_relevant)
print(f"dense (item=enriched, query=plain) Recall@50: {dense_recall:.4f}")

item-эмбеды (enriched): (344825, 768)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dense (item=enriched, query=plain) Recall@50: 0.1777


## BM25 (для примеров кандидатов ниже)

In [3]:
items_corpus = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)
items_corpus["text"] = (
    items_corpus["item_title_raw"] + " "
    + items_corpus["item_description_raw"] + " "
    + items_corpus["item_infm_params_text"]
)
corpus_item_ids_bm25 = items_corpus["item_id"].tolist()

BM25_CACHE = "../data/bm25s_index"
retriever = bm25s.BM25.load(BM25_CACHE, load_corpus=False)

val_tokens = val_queries["search_query"].apply(tokenize).tolist()
results, scores = retriever.retrieve(
    val_tokens, corpus=corpus_item_ids_bm25, k=50, show_progress=False, n_threads=0
)
bm25_predictions = {
    cid: list(results[i])
    for i, cid in enumerate(val_queries["context_id"])
}
bm25_recall = recall_at_50(bm25_predictions, val_relevant)
print(f"BM25 Recall@50: {bm25_recall:.4f}")

BM25 Recall@50: 0.1861


## Точечные (структурные) фичи для будущего реранкера

Не каналы генерации кандидатов, а признаки для пары (query, candidate item) - понадобятся, когда будем обучать CatBoost/сравнивать с cross-encoder. Всё посчитано на `train_part` (без утечки из val) там, где это влияет на статистику (популярность, центроид локации).

**Гео-дистанция** - самый сильный найденный сигнал.

In [4]:
from preprocessing import extract_label_value

train_part = train_part.copy()
train_part["item_latitude"] = train_part["item_latitude"].astype(float)
train_part["item_longitude"] = train_part["item_longitude"].astype(float)

loc_centroid = train_part.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()

item_meta = train.drop_duplicates("item_id").set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden",
    "item_infm_params_text",
]].copy()
item_meta["item_latitude"] = item_meta["item_latitude"].astype(float)
item_meta["item_longitude"] = item_meta["item_longitude"].astype(float)
item_meta["item_price"] = item_meta["item_price"].astype(float).replace(-1, np.nan)

item_popularity = train_part["item_id"].value_counts()


def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def build_pointwise_features(query_row, item_ids):
    meta = item_meta.reindex(item_ids)

    search_loc = query_row["search_location_id"]
    if search_loc in loc_centroid.index:
        c_lat, c_lon = loc_centroid.loc[search_loc, ["item_latitude", "item_longitude"]]
        geo_distance_km = haversine(c_lat, c_lon, meta["item_latitude"], meta["item_longitude"])
    else:
        geo_distance_km = pd.Series(np.nan, index=item_ids)
    geo_exact_match = (meta["item_location_id"] == search_loc).astype(int)

    q_vid = extract_label_value(query_row["search_infm_params_text"], "Вид услуги")
    q_tip = extract_label_value(query_row["search_infm_params_text"], "Тип услуги")
    item_vid = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
    item_tip = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))

    features = pd.DataFrame({
        "item_id": item_ids,
        "geo_distance_km": geo_distance_km.to_numpy(),
        "geo_exact_match": geo_exact_match.to_numpy(),
        "vid_uslugi_match": (item_vid == q_vid).astype(int).to_numpy() if q_vid else np.nan,
        "tip_uslugi_match": (item_tip == q_tip).astype(int).to_numpy() if q_tip else np.nan,
        "item_price": meta["item_price"].to_numpy(),
        "item_rating": meta["item_rating"].to_numpy(),
        "item_rating_reviews_count": meta["item_rating_reviews_count"].to_numpy(),
        "item_is_phone_hidden": meta["item_is_phone_hidden"].astype(int).to_numpy(),
        "item_is_message_forbidden": meta["item_is_message_forbidden"].astype(int).to_numpy(),
        "item_popularity": [item_popularity.get(iid, 0) for iid in item_ids],
    })
    return features

Проверим на одном реальном запросе с несколькими кандидатами (BM25 top-10 + истинный item):

In [5]:
sample_query = val_queries.iloc[0]
sample_relevant = val_relevant.get(sample_query["context_id"], [])
sample_candidates = list(dict.fromkeys(
    bm25_predictions.get(sample_query["context_id"], [])[:10] + list(sample_relevant)
))

print("query:", sample_query["search_query"], "| фильтры:", sample_query["search_infm_params_text"][:80])
print("истинный item(ы):", sample_relevant)
print()
feats = build_pointwise_features(sample_query, sample_candidates)
feats["is_relevant"] = feats["item_id"].isin(sample_relevant).astype(int)
feats

query: наращивание ногтей | фильтры: Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье
истинный item(ы): {'78c2899bda9f2375'}



,item_id,geo_distance_km,geo_exact_match,vid_uslugi_match,tip_uslugi_match,item_price,item_rating,item_rating_reviews_count,item_is_phone_hidden,item_is_message_forbidden,item_popularity,is_relevant
0,3b107bd05ba8ba30,NaN,0,1,1,2700.0,5.000000,3.0,0,0,4,0
1,eb358996cd133e06,NaN,0,1,1,500.0,4.931034,58.0,1,0,3,0
2,ba0358b46cc24dcd,NaN,0,1,1,1500.0,4.826772,127.0,1,0,3,0
3,417bf8b790865877,NaN,0,1,1,1000.0,4.906977,43.0,1,0,1,0
4,c9df5512858ac05f,NaN,0,1,1,2000.0,4.000000,11.0,1,0,1,0
5,f5752c9b70640f65,NaN,0,1,1,1500.0,4.826772,127.0,1,0,2,0
6,69009adb27b2a7a6,NaN,0,1,1,1000.0,5.000000,39.0,1,0,4,0
7,2f04d6a752a10f07,NaN,0,1,1,500.0,4.714286,7.0,0,0,2,0
8,4fa19e069b95169e,NaN,0,1,1,600.0,5.000000,11.0,0,0,1,0
9,75a5dfb0c51b6b8d,NaN,0,1,1,1500.0,4.800000,20.0,1,0,1,0


## Расширенный пул кандидатов + обученный реранкер (CatBoost)

Диагностировали структурную проблему: naive union просто конкатенирует top-50 каждого канала (BM25 первым) и режет на 50 ДО всякого ранжирования — BM25 почти всегда сам даёт полные 50 кандидатов, поэтому `union_recall == bm25_recall` буквально всегда, другие каналы не успевают попасть в финальные 50.

In [6]:
from collections import defaultdict

from catboost import CatBoostRanker, Pool

from eval.metrics import recall_at_k
from pipeline.geo import build_geo_tree, haversine
from preprocessing import extract_label_value

title_words = {
    iid: set(t.lower().split())
    for iid, t in zip(items_corpus["item_id"], items_corpus["item_title_raw"])
}

geo_tree, geo_item_ids = build_geo_tree(
    train.drop_duplicates("item_id")[["item_id", "item_location_id", "item_latitude", "item_longitude"]]
)

item_meta = train.drop_duplicates("item_id").set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden",
    "item_infm_params_text",
]].copy()
item_meta["item_latitude"] = item_meta["item_latitude"].astype(float)
item_meta["item_longitude"] = item_meta["item_longitude"].astype(float)
item_meta["item_price"] = item_meta["item_price"].astype(float).replace(-1, np.nan)

item_vid_series = item_meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
item_tip_series = item_meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))


In [7]:
from pipeline.geo import build_loc_centroid


def build_value_lookup(value_series, popularity):
    lookup = defaultdict(list)
    for item_id, value in value_series.items():
        if value:
            lookup[value].append(item_id)
    for value in lookup:
        lookup[value].sort(key=lambda iid: -popularity.get(iid, 0))
    return lookup


def build_stat_tables(stat_part):
    def build_lookup(item_ids, keys):
        lookup = defaultdict(dict)
        for key, item_id in zip(keys, item_ids):
            d = lookup[key]
            d[item_id] = d.get(item_id, 0) + 1
        return lookup

    context_train = stat_part[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
    query_norm_train = stat_part["search_query"].str.lower().str.strip()
    context_lookup = build_lookup(stat_part["item_id"], context_train)
    text_lookup = build_lookup(stat_part["item_id"], query_norm_train)
    item_popularity = stat_part["item_id"].value_counts()

    loc_centroid = build_loc_centroid(stat_part)

    vid_lookup = build_value_lookup(item_vid_series, item_popularity)
    tip_lookup = build_value_lookup(item_tip_series, item_popularity)

    location_to_items = defaultdict(list)
    for iid, loc in item_meta["item_location_id"].items():
        location_to_items[loc].append(iid)
    for loc in location_to_items:
        location_to_items[loc].sort(key=lambda iid: -item_popularity.get(iid, 0))

    return context_lookup, text_lookup, item_popularity, loc_centroid, vid_lookup, tip_lookup, location_to_items


def get_pool_and_channels(query_df, context_lookup, text_lookup, loc_centroid, vid_lookup, tip_lookup, location_to_items, k=100, batch=2000):
    all_pools, all_channels = {}, {}
    for start in range(0, len(query_df), batch):
        qdf = query_df.iloc[start:start + batch].reset_index(drop=True)
        n = len(qdf)
        val_tokens = qdf["search_query"].apply(tokenize).tolist()
        bm25_results, bm25_scores = retriever.retrieve(
            val_tokens, corpus=corpus_item_ids_bm25, k=k, show_progress=False, n_threads=0
        )
        query_texts = qdf["search_query"].apply(clean_text).apply(lambda t: f"query: {t}").tolist()
        q_emb = embed_model.encode(query_texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        dense_scores_all, dense_idx_all = index.search(q_emb, k)

        query_coords = loc_centroid.reindex(qdf["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
        valid_mask = ~np.isnan(query_coords).any(axis=1)
        geo_dist_all = np.full((n, k), np.nan)
        geo_idx_all = np.full((n, k), -1, dtype=int)
        if valid_mask.any():
            d, idx = geo_tree.query(np.radians(query_coords[valid_mask]), k=k)
            geo_dist_all[valid_mask] = d * 6371.0
            geo_idx_all[valid_mask] = idx

        contexts = qdf[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1).tolist()
        qnorms = qdf["search_query"].str.lower().str.strip().tolist()
        search_params_list = qdf["search_infm_params_text"].tolist()
        search_loc_list = qdf["search_location_id"].tolist()

        for i, cid in enumerate(qdf["context_id"]):
            bm25_ch = {bm25_results[i][r]: (r + 1, float(bm25_scores[i][r])) for r in range(k)}
            dense_ch = {dense_item_ids[dense_idx_all[i][r]]: (r + 1, float(dense_scores_all[i][r])) for r in range(k) if dense_idx_all[i][r] != -1}

            scores = {}
            for item_id, cnt in context_lookup.get(contexts[i], {}).items():
                scores[item_id] = scores.get(item_id, 0) + cnt * 2
            for item_id, cnt in text_lookup.get(qnorms[i], {}).items():
                scores[item_id] = scores.get(item_id, 0) + cnt
            ranked_lookup = sorted(scores.items(), key=lambda x: -x[1])[:k]
            lookup_ch = {iid: (r + 1, s) for r, (iid, s) in enumerate(ranked_lookup)}

            geo_ch = {}
            if geo_idx_all[i][0] != -1:
                for r in range(k):
                    gi = geo_idx_all[i][r]
                    if gi == -1:
                        continue
                    geo_ch[geo_item_ids[gi]] = (r + 1, float(geo_dist_all[i][r]))

            tip = extract_label_value(search_params_list[i], "Тип услуги")
            vid = extract_label_value(search_params_list[i], "Вид услуги")
            filter_cands = tip_lookup.get(tip, [])[:k] if tip else (vid_lookup.get(vid, [])[:k] if vid else [])
            filter_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(filter_cands)}

            geo_exact_cands = location_to_items.get(search_loc_list[i], [])[:k]
            geo_exact_ch = {iid: (r + 1, float(item_popularity.get(iid, 0))) for r, iid in enumerate(geo_exact_cands)}

            pool = list(dict.fromkeys(
                list(bm25_ch.keys()) + list(dense_ch.keys()) + list(lookup_ch.keys()) + list(geo_ch.keys())
                + list(filter_ch.keys()) + list(geo_exact_ch.keys())
            ))
            all_pools[cid] = pool
            all_channels[cid] = {
                "bm25": bm25_ch, "dense": dense_ch, "lookup": lookup_ch, "geo": geo_ch,
                "filter": filter_ch, "geo_exact": geo_exact_ch,
            }
    return all_pools, all_channels


FEATURE_COLS = [
    "geo_distance_km", "geo_exact_match", "vid_uslugi_match", "tip_uslugi_match",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden", "item_popularity", "text_overlap",
    "bm25_rank", "bm25_present", "bm25_score",
    "dense_rank", "dense_present", "dense_score",
    "lookup_rank", "lookup_present", "lookup_score",
    "geo_rank", "geo_present", "geo_score",
    "filter_rank", "filter_present", "filter_score",
    "geo_exact_rank", "geo_exact_present", "geo_exact_score",
]


def assemble_features(query_df, pools, channels, item_popularity, loc_centroid):
    rows = []
    query_rows = {row["context_id"]: row for _, row in query_df.iterrows()}
    for cid, pool in pools.items():
        if not pool:
            continue
        qrow = query_rows[cid]
        ch = channels[cid]
        meta = item_meta.reindex(pool)

        search_loc = qrow["search_location_id"]
        if search_loc in loc_centroid.index:
            c_lat, c_lon = loc_centroid.loc[search_loc, ["item_latitude", "item_longitude"]]
            geo_distance_km = haversine(c_lat, c_lon, meta["item_latitude"], meta["item_longitude"]).to_numpy()
        else:
            geo_distance_km = np.full(len(pool), np.nan)
        geo_exact_match = (meta["item_location_id"] == search_loc).astype(int).to_numpy()

        q_vid = extract_label_value(qrow["search_infm_params_text"], "Вид услуги")
        q_tip = extract_label_value(qrow["search_infm_params_text"], "Тип услуги")
        item_vid = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
        item_tip = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))

        q_words = set(qrow["search_query"].lower().split())
        text_overlap = np.array([len(q_words & title_words.get(iid, set())) for iid in pool])

        df = pd.DataFrame({
            "context_id": cid,
            "item_id": pool,
            "geo_distance_km": geo_distance_km,
            "geo_exact_match": geo_exact_match,
            "vid_uslugi_match": (item_vid == q_vid).astype(int).to_numpy() if q_vid else np.nan,
            "tip_uslugi_match": (item_tip == q_tip).astype(int).to_numpy() if q_tip else np.nan,
            "item_price": meta["item_price"].to_numpy(),
            "item_rating": meta["item_rating"].to_numpy(),
            "item_rating_reviews_count": meta["item_rating_reviews_count"].to_numpy(),
            "item_is_phone_hidden": meta["item_is_phone_hidden"].astype(int).to_numpy(),
            "item_is_message_forbidden": meta["item_is_message_forbidden"].astype(int).to_numpy(),
            "item_popularity": [item_popularity.get(iid, 0) for iid in pool],
            "text_overlap": text_overlap,
        })
        for name, cmap in ch.items():
            ranks = np.array([cmap.get(iid, (0, np.nan))[0] for iid in pool])
            scores = np.array([cmap.get(iid, (0, np.nan))[1] for iid in pool])
            df[f"{name}_rank"] = ranks
            df[f"{name}_present"] = (ranks > 0).astype(int)
            df[f"{name}_score"] = scores
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


### Обучающие пары для CatBoost

In [8]:
fit_part, cb_train_queries, cb_train_relevant = make_val_split(train_part, val_size=0.05, random_state=7)

SAMPLE_N = 8000
cb_train_queries = cb_train_queries.sample(min(SAMPLE_N, len(cb_train_queries)), random_state=1).reset_index(drop=True)
print("fit_part:", len(fit_part), "cb_train_queries (сэмпл):", len(cb_train_queries))

cb_context_lookup, cb_text_lookup, cb_item_popularity, cb_loc_centroid, cb_vid_lookup, cb_tip_lookup, cb_location_to_items = build_stat_tables(fit_part)

pools_train, channels_train = get_pool_and_channels(
    cb_train_queries, cb_context_lookup, cb_text_lookup, cb_loc_centroid, cb_vid_lookup, cb_tip_lookup, cb_location_to_items, k=100
)
print("средний размер пула (train):", np.mean([len(p) for p in pools_train.values()]))

feat_train = assemble_features(cb_train_queries, pools_train, channels_train, cb_item_popularity, cb_loc_centroid)
feat_train["label"] = feat_train.apply(
    lambda r: int(r["item_id"] in cb_train_relevant.get(r["context_id"], set())), axis=1
)
feat_train = feat_train.sort_values("context_id").reset_index(drop=True)
print("feat_train:", feat_train.shape, "позитивов:", feat_train["label"].sum())


fit_part: 425717 cb_train_queries (сэмпл): 8000
средний размер пула (train): 418.758
feat_train: (3350064, 32) позитивов: 5221


### Обучение CatBoostRanker (PairLogit)

In [9]:
X = feat_train[FEATURE_COLS]
y = feat_train["label"]
group_id = feat_train["context_id"]

uniq_ctx = feat_train["context_id"].unique()
rng = np.random.RandomState(0)
holdout_ctx = set(rng.choice(uniq_ctx, size=int(len(uniq_ctx) * 0.1), replace=False))
train_mask = ~feat_train["context_id"].isin(holdout_ctx)

train_pool = Pool(X[train_mask], y[train_mask], group_id=group_id[train_mask])
eval_pool = Pool(X[~train_mask], y[~train_mask], group_id=group_id[~train_mask])

catboost_model = CatBoostRanker(
    loss_function="PairLogit", iterations=300, learning_rate=0.1, depth=6,
    random_seed=42, verbose=50,
)
catboost_model.fit(train_pool, eval_set=eval_pool)

importances = catboost_model.get_feature_importance(train_pool)
pd.DataFrame({"feature": FEATURE_COLS, "importance": importances}).sort_values("importance", ascending=False)

0:	learn: 0.3902459	test: 0.3924742	best: 0.3924742 (0)	total: 629ms	remaining: 3m 8s
50:	learn: 0.0379951	test: 0.0518765	best: 0.0516704 (39)	total: 28.7s	remaining: 2m 20s
100:	learn: 0.0297938	test: 0.0496355	best: 0.0493517 (95)	total: 54.3s	remaining: 1m 46s
150:	learn: 0.0248927	test: 0.0497974	best: 0.0488538 (146)	total: 1m 22s	remaining: 1m 21s
200:	learn: 0.0203195	test: 0.0516864	best: 0.0486069 (164)	total: 1m 52s	remaining: 55.2s
250:	learn: 0.0174855	test: 0.0518696	best: 0.0486069 (164)	total: 2m 29s	remaining: 29.2s
299:	learn: 0.0153391	test: 0.0523777	best: 0.0486069 (164)	total: 3m 6s	remaining: 0us

bestTest = 0.0486069094
bestIteration = 164

Shrink model to first 165 iterations.


,feature,importance
0,geo_distance_km,0.027878
9,item_popularity,0.025092
28,geo_exact_score,0.007773
1,geo_exact_match,0.007402
10,text_overlap,0.004937
2,vid_uslugi_match,0.003805
13,bm25_score,0.003270
16,dense_score,0.001473
11,bm25_rank,0.000748
15,dense_present,0.000674


### Оценка на val

Статистики каналов для val - из `train_part` целиком (как везде в проекте), не из `fit_part` (тот был только для отделения обучающих пар ранкера).

In [10]:
context_lookup, text_lookup, item_popularity, loc_centroid, vid_lookup, tip_lookup, location_to_items = build_stat_tables(train_part)

In [11]:
pools_val, channels_val = get_pool_and_channels(
    val_queries, context_lookup, text_lookup, loc_centroid, vid_lookup, tip_lookup, location_to_items, k=100
)
avg_pool_val = np.mean([len(p) for p in pools_val.values()])
print("средний размер пула (val):", avg_pool_val)

feat_val = assemble_features(val_queries, pools_val, channels_val, item_popularity, loc_centroid)
feat_val["score"] = catboost_model.predict(feat_val[FEATURE_COLS])

feat_val_sorted = feat_val.sort_values(["context_id", "score"], ascending=[True, False])
catboost_predictions = feat_val_sorted.groupby("context_id")["item_id"].apply(lambda s: list(s)[:50]).to_dict()

catboost_recall = recall_at_50(catboost_predictions, val_relevant)
print(f"CatBoost реранкер Recall@50: {catboost_recall:.4f}")

naive_pool_predictions = {cid: pool[:50] for cid, pool in pools_val.items()}
naive_pool_recall = recall_at_50(naive_pool_predictions, val_relevant)
print(f"Тот же пул без ранкера (первые 50 по порядку каналов) Recall@50: {naive_pool_recall:.4f}")

max_pool_size = max(len(p) for p in pools_val.values())
pool_ceiling_recall = recall_at_k(pools_val, val_relevant, k=max_pool_size)
print(f"Потолок пула (recall@{max_pool_size}, без обрезки) : {pool_ceiling_recall:.4f}")
print("Лучший RRF-бейзлайн без ранкера: 0.1940")

средний размер пула (val): 420.9122069568652
CatBoost реранкер Recall@50: 0.5126
Тот же пул без ранкера (первые 50 по порядку каналов) Recall@50: 0.1861
Потолок пула (recall@595, без обрезки) : 0.5365
Лучший RRF-бейзлайн без ранкера: 0.1940
